# 29. Function decorators and method binding

**Level:** Intermediate

**Learning outcome:** Trace how functions become bound methods and write decorators that preserve results and metadata.

**Prerequisites:** 02, 04, and 16: method types, dependencies, and decorator objects.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](28_Equality_Hashing_and_Operator_Overloading.ipynb) | [Next notebook](30_Iterators_Generators_and_Lazy_Object_APIs.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Mental model
A decorator transforms an object at definition time. For a function, `@trace` above `def read(...)`
is approximately `read = trace(read)` after the function is created. A decorator can return a
wrapper that does work before or after calling the original. It must return the original result
and preserve error behavior unless its documented purpose says otherwise.

Function decorators and decorator objects solve related but different problems. A transport
wrapper object exposes send and holds another transport; a function decorator replaces a callable.
`functools.wraps` copies useful metadata and supplies __wrapped__ for introspection. It does not
make a synchronous wrapper async-aware automatically.

Functions defined on a class implement descriptor behavior. Access through an instance normally
produces a bound method carrying the instance. Python supplies that instance as the first
argument. A classmethod binds the class instead; a staticmethod supplies neither. These facts
explain why a wrapper using *args and **kwargs can decorate an ordinary instance method.

Keep instrumentation local and avoid printing full arguments that could contain credentials.
A counter around an attempted operation counts attempts unless you increment only after success.

## Worked example 1: Inspect a bound method

**Problem:** See exactly which object Python supplies as self.

### Step-by-step explanation
1. Define one ordinary instance method with an explicit extra parameter.
2. Retrieve the method without calling it to inspect the binding.
3. Compare the bound call with the explicit class-level function call.
4. Observe the same returned value from both forms.

**Expected output:**
```text
device:edge-01
```

**Common mistake / limitation:** Do not pass self again to a bound method; it has already been supplied.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [1]:
class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def label(self, prefix):
        return prefix + self.hostname

device = Device("edge-01")
bound = device.label
assert bound.__self__ is device
assert bound.__func__ is Device.label
assert bound("device:") == Device.label(device, "device:")
print(bound("device:"))

device:edge-01


### Practice 1

Inspect a classmethod's __self__ and explain why it refers to the class.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 2: Count attempts with a method decorator

**Problem:** Instrument a read without losing its return value or function metadata.

### Step-by-step explanation
1. The decorator returns a new function that will still bind normally on the class.
2. The wrapper receives self and updates per-instance state.
3. Delegate all other arguments and return the wrapped result unchanged.
4. Verify count, result, and metadata rather than checking only printed text.

**Expected output:**
```text
attempts: 2 name: read
```

**Common mistake / limitation:** This decorator requires self.attempts by contract; document that dependency rather than hiding it.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [2]:
from functools import wraps

def count_attempts(function):
    @wraps(function)
    def wrapper(self, *args, **kwargs):
        self.attempts += 1
        return function(self, *args, **kwargs)
    return wrapper

class Reader:
    def __init__(self):
        self.attempts = 0

    @count_attempts
    def read(self, interface):
        """Return an offline interface label."""
        return interface + ":up"

reader = Reader()
assert reader.read("Ethernet1/1") == "Ethernet1/1:up"
assert reader.read("Ethernet1/2") == "Ethernet1/2:up"
assert reader.attempts == 2
assert Reader.read.__name__ == "read"
assert Reader.read.__wrapped__.__doc__ == "Return an offline interface label."
print("attempts:", reader.attempts, "name:", Reader.read.__name__)

attempts: 2 name: read


### Practice 2

Add separate success and failure counts while re-raising the original error.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 3: Parameterized validation decorator

**Problem:** Return a decorator from a factory to configure one declared policy.

### Step-by-step explanation
1. The outer factory captures low and high in a closure.
2. The middle function receives and transforms the method definition.
3. The wrapper validates before calling a mutating method.
4. After a rejected update, verify the old value is still present.

**Expected output:**
```text
invalid update rejected before mutation
9000
```

**Common mistake / limitation:** For one field, a property may be clearer. A decorator is not automatically the best validation mechanism.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [3]:
from functools import wraps

def bounded_mtu(low, high):
    def decorate(function):
        @wraps(function)
        def wrapper(self, mtu):
            if type(mtu) is not int or not low <= mtu <= high:
                raise ValueError("MTU outside teaching policy")
            return function(self, mtu)
        return wrapper
    return decorate

class Interface:
    def __init__(self):
        self.mtu = 1500

    @bounded_mtu(576, 9216)
    def set_mtu(self, mtu):
        self.mtu = mtu
        return mtu

interface = Interface()
assert interface.set_mtu(9000) == 9000
try:
    interface.set_mtu(0)
except ValueError:
    assert interface.mtu == 9000
    print("invalid update rejected before mutation")
else:
    raise AssertionError("invalid MTU accepted")
print(interface.mtu)

invalid update rejected before mutation
9000


### Practice 3

Refactor the lesson into a validated property and compare readability.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Check your understanding: Function decorators and method binding

**You should now be able to:** Trace how functions become bound methods and write decorators that preserve results and metadata.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/library/functools.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.